## SQL Basics

In [0]:
-- saving catalog and schema 
use catalog workspace;
use schema identifier('default');
-- viewing schema and catalog
select 
  current_catalog(),
  current_schema()

In [0]:
-- command to view available schemas in the catalog
show schemas in workspace;

In [0]:
-- To get the information about the schema (describe schema extended identifier('schema_name')- this is incase you've stored schema name in a var)
describe schema extended default

In [0]:
-- to get the information about the tables
describe table extended sample_table

In [0]:
-- describa a volume
describe volume volume_intro;

In [0]:
-- list files in a volume
list '/Volumes/workspace/default/volume_intro'

In [0]:
-- query the parquet files by path in the directory to view raw data as a quick preview in table format
select * 
from parquet.`/Volumes/workspace/v01/parquet`
limit 5;

In [0]:
-- Read parquet data with read_files
select *
from read_files('/Volumes/workspace/v01/parquet',format=> 'parquet')
limit 5;

## Batch Data Ingestion

In [0]:
--Drop the table if already exists
drop table if exists parquet_bronze_ctas;

-- create the table
create table parquet_bronze_ctas
select *
from read_files('/Volumes/workspace/v01/parquet',format=> 'parquet');

--preview the delta table
select * from parquet_bronze_ctas
limit 5;


In [0]:
describe table extended parquet_bronze_ctas;

In [0]:
%python
# Data Batch Ingestion with Python
# read parquet files in spark df
df = spark.read.format('parquet').load('/Volumes/workspace/v01/parquet')
# write the df to a delta table
df.write.mode('overwrite').saveAsTable('workspace.default.parquet_bronze_python')
# read and view the table
parque_bronze_table=spark.table('workspace.default.parquet_bronze_python')
parque_bronze_table.limit(5).display()

## Incremental ingestion using copy into

In [0]:
-- copy into schema mismatch
-- This will throw an error because there's mimatch between the schema of the table and the schema of the parquet data 
drop table if exists parquet_bronze_ci;
-- create an empty table with specified schema
create table parquet_bronze_ci(
  first_name string,
  last_name string
);
-- use copy into to populate delta table
copy into parquet_bronze_ci
from '/Volumes/workspace/v01/parquet'
fileformat = parquet;

In [0]:
-- we can resolve the above error with merge schema=True
copy into parquet_bronze_ci
from '/Volumes/workspace/v01/parquet'
fileformat = parquet
copy_options('mergeSchema' = 'true');


In [0]:
select *
from parquet_bronze_ci
limit 5;
-- note what's the order we've created in the first time the same order it'll merge the final table

In [0]:
-- we can also create with empty table first and merge the schema
drop table if exists parquet_bronze_ci_no_schema;

create table parquet_bronze_ci_no_schema;

copy into parquet_bronze_ci_no_schema
  from '/Volumes/workspace/v01/parquet'
  fileformat = parquet
  copy_options('mergeschema'='true');

In [0]:
select *
  from parquet_bronze_ci_no_schema
  limit 5;

In [0]:
--to check the idempotentency
-- as this is the incremental already all files are already loaded it won't load anything
copy into parquet_bronze_ci_no_schema
  from '/Volumes/workspace/v01/parquet'
  fileformat = parquet
  copy_options('mergeschema'='true');

## Adding metadata during ingestion

In [0]:
-- altering the data type of a column while reading the files in a volume
select *,try_cast(cc as bigint) as cc_num
from read_files('/Volumes/workspace/v01/parquet',format=> 'parquet')
limit 5;

In [0]:
-- convert unix timestamp to date
select *,try_cast(cc as bigint) as cc_num,cast(from_unixtime(cc_num/1000000) as date) as cc_date
from read_files('/Volumes/workspace/v01/parquet',format=> 'parquet')
limit 5;

In [0]:
-- adding column metadata on ingestion
select *,
  try_cast(cc as bigint) as cc_num,
  cast(from_unixtime(cc_num/1000000) as date) as cc_date,
  _metadata.file_modification_time as file_modification_time,
  _metadata.file_name as source_file,
  current_timestamp() as ingestion_time
from read_files('/Volumes/workspace/v01/parquet',format=> 'parquet')
limit 5;

In [0]:
drop table if exists parquet_bronze;

create table parquet_bronze as
select *,
  try_cast(cc as bigint) as cc_num,
  cast(from_unixtime(cc_num/1000000)as date) as cc_date,
  _metadata.file_modification_time as file_modification_time,
  _metadata.file_name as source_file,
  current_timestamp() as ingestion_time
from read_files("/Volumes/workspace/v01/parquet",format=>'parquet');

select *
  from parquet_bronze
  limit 5;

In [0]:
-- exploring the metadata info from the table
select source_file,count(*) as total
  from parquet_bronze
  group by source_file
  order by source_file;